In [1]:
import pandas as pd
import numpy as np

In [2]:
df_raw = pd.read_csv('../../dataset/original/billings.csv')
print(len(df_raw.columns))
print(df_raw.info())
print(df_raw.head())

C:\Users\Asus\AppData\Local\Temp\ipykernel_40616\2992768314.py:1: DtypeWarning: Columns (15,16,19,52,53) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv('../../dataset/original/billings.csv')


59
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122082 entries, 0 to 122081
Data columns (total 59 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Co_Ref                      122082 non-null  object 
 1   Renewal_Month               122082 non-null  object 
 2   Connection_Net              8037 non-null    float64
 3   Connection_Qty              8037 non-null    float64
 4   Discount_Amount             13551 non-null   object 
 5   Sustainability_Score        122082 non-null  float64
 6   Total_Renewal_Score_New     122082 non-null  float64
 7   Starting_Connection_Net     8545 non-null    float64
 8   Starting_Connection_Qty     8545 non-null    float64
 9   Last_Years_Price            112992 non-null  float64
 10  Last_Years_Date_Paid        0 non-null       float64
 11  Auto_Renewal_Score          122082 non-null  int64  
 12  Status_Scores               122082 non-null  int64  
 13  Anchoring_S

In [3]:
df_cols = df_raw.columns
print(df_cols)
# //save as a csv file
df_cols_df = pd.DataFrame(df_cols, columns=['Columns'])
df_cols_df.to_csv('../../dataset/01_raw/billings/columns.csv', index=True)

Index(['Co_Ref', 'Renewal_Month', 'Connection_Net', 'Connection_Qty',
       'Discount_Amount', 'Sustainability_Score', 'Total_Renewal_Score_New',
       'Starting_Connection_Net', 'Starting_Connection_Qty',
       'Last_Years_Price', 'Last_Years_Date_Paid', 'Auto_Renewal_Score',
       'Status_Scores', 'Anchoring_Score', 'Tenure_Scores',
       'Proforma_Auto_Renewal', 'Proforma_World_Pay_Token', 'Proforma_Date',
       'Current_Anchorings', 'Current_Anchor_List', 'Payment_Timeframe',
       'Registration_Date', 'Proforma_Account_Stage', 'Proforma_Audit_Status',
       'Current_Auto_Renewal_Flag', 'Current_World_Pay_Token',
       'Renewal_Score_At_Release', 'Proforma_Membership_Status',
       'Proforma_Approved_Lists', 'Tenure_Years', 'Band',
       'Prospect_Renewal_Date', 'Closed_Date', 'Prospect_Status',
       'Starting_Net', 'Starting_Vat', 'Starting_Gross',
       'Starting_Membership_Net', 'Starting_Package_Net', 'Starting_PQQ_Net',
       'Gross', 'Membership_Net', 'Package_

## Analysing the Null Cols

In [4]:
print(df_raw.isnull().sum())

Co_Ref                             0
Renewal_Month                      0
Connection_Net                114045
Connection_Qty                114045
Discount_Amount               108531
Sustainability_Score               0
Total_Renewal_Score_New            0
Starting_Connection_Net       113537
Starting_Connection_Qty       113537
Last_Years_Price                9090
Last_Years_Date_Paid          122082
Auto_Renewal_Score                 0
Status_Scores                      0
Anchoring_Score                    0
Tenure_Scores                      0
Proforma_Auto_Renewal          18052
Proforma_World_Pay_Token       18052
Proforma_Date                    304
Current_Anchorings                 0
Current_Anchor_List            25957
Payment_Timeframe              20856
Registration_Date               1018
Proforma_Account_Stage          9229
Proforma_Audit_Status           9229
Current_Auto_Renewal_Flag          0
Current_World_Pay_Token            0
Renewal_Score_At_Release         126
P

In [5]:
null_columns = df_raw.columns[df_raw.isnull().any()]
print("Columns with null values:", null_columns)

#save the null cols as a csv file
null_cols_df = pd.DataFrame(null_columns, columns=['Null Columns']) 
null_cols_df.to_csv('../../dataset/01_raw/billings/null_columns.csv', index=True)


Columns with null values: Index(['Connection_Net', 'Connection_Qty', 'Discount_Amount',
       'Starting_Connection_Net', 'Starting_Connection_Qty',
       'Last_Years_Price', 'Last_Years_Date_Paid', 'Proforma_Auto_Renewal',
       'Proforma_World_Pay_Token', 'Proforma_Date', 'Current_Anchor_List',
       'Payment_Timeframe', 'Registration_Date', 'Proforma_Account_Stage',
       'Proforma_Audit_Status', 'Renewal_Score_At_Release',
       'Proforma_Membership_Status', 'Proforma_Approved_Lists', 'Tenure_Years',
       'Band', 'Closed_Date', 'Total_Net_Paid', 'Connection_Group',
       'Tenure_Group', '#_of_Connection', 'Last_Renewal', 'Last_Band',
       'Last_Total_Net_Paid', 'Last_Connections', 'Anchor_Group'],
      dtype='object')


## Getting the datatype of each columns and uniques values if (more than 10 values then as ranges)

In [6]:



def is_date_column(series):
    return pd.api.types.is_datetime64_any_dtype(series)


def is_coref_column(series, threshold=50):
    num_unique = series.nunique(dropna=True)
    return series.dtype == 'object' and num_unique > threshold


def classify_column(series):
    if pd.api.types.is_datetime64_any_dtype(series):
        return "datetime"

    if pd.api.types.is_numeric_dtype(series):
        return "numerical"

    if series.dtype == 'object':
        # Try parsing to datetime
        try:
            parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
            if all(parsed.dt.time == pd.Timestamp("00:00:00").time()):
                return "date"
            else:
                return "datetime"
        except:
            pass

        # Try parsing to time
        try:
            pd.to_datetime(series.dropna().head(10), format='%H:%M:%S', errors='raise')
            return "time"
        except:
            pass

        return "categorical"

    return "categorical"


def get_unique_representation(series):
    unique_vals = series.dropna().unique()
    num_unique = len(unique_vals)

    if is_date_column(series) or is_coref_column(series):
        return []

    if pd.api.types.is_numeric_dtype(series):
        if num_unique <= 10:
            return sorted(unique_vals.tolist())
        else:
            return f"{series.min()} - {series.max()}"

    return unique_vals.tolist()


def create_data_validation_df(df):
    summary = []

    for col in df.columns:
        series = df[col]

        col_summary = {
            "column_name": col,
            "data_type": str(series.dtype),
            "column_type": classify_column(series),
            "unique_values": get_unique_representation(series),
            "null_count": series.isna().sum(),
            "num_unique": series.nunique(dropna=True)
        }

        summary.append(col_summary)

    return pd.DataFrame(summary)


def save_validation_report(df, output_path):
    dt_val_df = create_data_validation_df(df)
    print(dt_val_df)

    dt_val_df.to_csv(output_path, index=True)
    print(f"\nReport saved to: {output_path}")


# output_file = './dataset/01_raw/ad_hoc/data_validation_summary.csv'
# save_validation_report(df_raw, output_file)

In [7]:
def create_data_validation_df(df):
    summary = []

    for col in df.columns:
        series = df[col]

        col_summary = {
            "column_name": col,
            "data_type": str(series.dtype),
            "column_type": classify_column(series),
            "null_count": series.isna().sum(),
            "null_percentage": round((series.isna().sum() / len(df)) * 100, 2),
            "num_unique": series.nunique(dropna=True),
            "unique_values": get_unique_representation(series),
            "top_values": series.value_counts(dropna=True).head(5).to_dict(),
            "skewness": series.skew() if pd.api.types.is_numeric_dtype(series) else None,
            "constant_column": series.nunique(dropna=True) <= 1
        }

        summary.append(col_summary)

    return pd.DataFrame(summary)

df_validation_df = create_data_validation_df(df_raw)
print(df_validation_df.head())
df_validation_df.to_csv('../../dataset/01_raw/billings/data_validation_summary.csv', index=True)




C:\Users\Asus\AppData\Local\Temp\ipykernel_40616\3920180566.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_40616\3920180566.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_40616\3920180566.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_

       column_name data_type  column_type  null_count  null_percentage  \
0           Co_Ref    object  categorical           0             0.00   
1    Renewal_Month    object         date           0             0.00   
2   Connection_Net   float64    numerical      114045            93.42   
3   Connection_Qty   float64    numerical      114045            93.42   
4  Discount_Amount    object  categorical      108531            88.90   

   num_unique                                      unique_values  \
0       47826                                                 []   
1          49  [01-11-2024, 01-08-2025, 01-03-2025, 01-06-202...   
2         161                                      55.0 - 4752.0   
3          48                                         1.0 - 48.0   
4          11  [35.00%, 20.00%, 30.00%, 40.00%, 10.00%, 50.00...   

                                          top_values  skewness  \
0  {'YG4840': 5, 'PU4860': 5, 'DK7141': 4, 'DZ463...       NaN   
1  {'01-03-202


## Mixed datatype cols

In [8]:
def check_mixed_data_types(df):
    mixed_summary = []

    for col in df.columns:
        series = df[col].dropna()

        # Try numeric conversion
        converted = pd.to_numeric(series, errors='coerce')

        # Count valid numeric vs invalid
        numeric_count = converted.notna().sum()
        non_numeric_count = converted.isna().sum()

        # If both exist → mixed
        if numeric_count > 0 and non_numeric_count > 0:
            sample_numeric = series[converted.notna()].iloc[0]
            sample_non_numeric = series[converted.isna()].iloc[0]

            mixed_summary.append({
                "column": col,
                "numeric_count": int(numeric_count),
                "non_numeric_count": int(non_numeric_count),
                "sample_numeric": sample_numeric,
                "sample_non_numeric": sample_non_numeric
            })

    return pd.DataFrame(mixed_summary)

mixed_df = check_mixed_data_types(df_raw)
print(mixed_df)

mixed_df.to_csv('../../dataset/01_raw/billings/mixed_data_types.csv', index=False)

             column  numeric_count  non_numeric_count sample_numeric  \
0  Connection_Group          68008              53948              1   
1      Tenure_Group          44737              76327              3   
2      Anchor_Group          68008              53948              1   

  sample_non_numeric  
0             4 to 9  
1                 4+  
2             4 to 9  


## =================================================================================